Machine-Learning Forecasting.

In [ ]:
from pathlib import Path
import pandas as pd

project_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "code" / "notebooks").is_dir()
    ),
    Path.cwd(),
)
data_dir = project_dir / "data" / "raw" / "m5-forecasting-accuracy"

print("Dataset folder found:", data_dir.exists(), data_dir)

for filename in [
    "calendar.csv",
    "sell_prices.csv",
    "sales_train_validation.csv",
]:
    sample = pd.read_csv(data_dir / filename, nrows=5)
    print(f"\n{filename}: preview of {sample.shape[0]} rows")
    print(sample.iloc[:2, :8].to_string(index=False))

In [ ]:
all_sales = pd.read_csv(data_dir / "sales_train_validation.csv")
all_sales.shape

In [ ]:
daily_sales_all = all_sales.iloc[:, 6:]
daily_sales_all.shape

In [ ]:
# Latest period: train through d_1885, validate on d_1886–d_1913
train_sales_all = daily_sales_all.iloc[:, :-28]
validation_sales_all = daily_sales_all.iloc[:, -28:]

# Earlier period 1: train through d_1857, validate on d_1858–d_1885
train_window_1 = daily_sales_all.iloc[:, :-56]
validation_window_1 = daily_sales_all.iloc[:, -56:-28]

# Earlier period 2: train through d_1829, validate on d_1830–d_1857
train_window_2 = daily_sales_all.iloc[:, :-84]
validation_window_2 = daily_sales_all.iloc[:, -84:-56]

(
    train_sales_all.shape, validation_sales_all.shape,
    train_window_1.shape, validation_window_1.shape,
    train_window_2.shape, validation_window_2.shape
)

In [ ]:
baseline_result_summary = pd.DataFrame({
    "baseline": [
        "7-day avg",
        "28-day avg",
        "weekday-aware avg",
        "repeat last week"
    ],
    "MAE_latest": [1.062837, 1.041132, 1.073854, 1.232355],
    "MAE_window_1": [1.048760, 1.045754, 1.083339, 1.210618],
    "MAE_window_2": [0.991200, 1.033281, 1.067838, 1.199449],
    "Mean_MAE": [1.034265, 1.040056, 1.075010, 1.214141]
})

baseline_result_summary

In [ ]:
import sklearn
sklearn.__version__

In [ ]:
sample_1000 = all_sales.sample(n=1000 , random_state=50)

daily_sales_1000 = sample_1000.iloc[: , 6:]

train_sales_1000_window_1 = daily_sales_1000.iloc[: , :-56]
validation_sales_1000_window_1 = daily_sales_1000.iloc[: , -56:-28]

train_sales_1000_window_1.shape , validation_sales_1000_window_1.shape

In [ ]:
example_series = train_sales_1000_window_1.iloc[0]

example_features = pd.DataFrame({"sales": example_series})
example_features["lag_1"] = example_features["sales"].shift(1)
example_features["lag_7"] = example_features["sales"].shift(7)
example_features["lag_28"] = example_features["sales"].shift(28)

example_features.iloc[25:32]

In [ ]:
sales_long_1000 = (
    train_sales_1000_window_1
    .rename_axis(index="series_id", columns="day")
    .stack()
    .rename("sales")
    .reset_index()
)

sales_long_1000.head()

In [ ]:
sales_long_1000["lag_1"] = (
    sales_long_1000.groupby("series_id")["sales"].shift(1)
)

sales_long_1000["lag_7"] = (
    sales_long_1000.groupby("series_id")["sales"].shift(7)
)

sales_long_1000["lag_28"] = (
    sales_long_1000.groupby("series_id")["sales"].shift(28)
)

sales_long_1000.iloc[25:32]

In [ ]:
modal_data_1000 = sales_long_1000.dropna(
    subset = ["lag_1","lag_7","lag_28"]
).copy()

x_train_1000 = modal_data_1000[["lag_1","lag_7","lag_28"]]
y_train_1000 = modal_data_1000["sales"]

x_train_1000.shape, y_train_1000.shape

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

model_1000 = HistGradientBoostingRegressor(
    max_iter=50,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=50
)

model_1000.fit(x_train_1000, y_train_1000)

In [ ]:
import numpy as np

history = train_sales_1000_window_1.to_numpy(dtype=float)
forecast_values = np.zeros((history.shape[0], 28))

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history[:, -1],
        "lag_7": history[:, -7],
        "lag_28": history[:, -28],
    })

    next_prediction = model_1000.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values[:, day_step] = next_prediction
    history = np.column_stack([history, next_prediction])

forecast_ml_window_1 = pd.DataFrame(
    forecast_values,
    index=validation_sales_1000_window_1.index,
    columns=validation_sales_1000_window_1.columns
)

forecast_ml_window_1.iloc[:5, :5]

In [ ]:
error_ml_window_1 = (
    validation_sales_1000_window_1 - forecast_ml_window_1
).abs()

mae_ml_window_1 = error_ml_window_1.mean().mean()
mae_ml_window_1

avg_last_7-days

In [ ]:
avg_last_7_sample_window_1 = (
    train_sales_1000_window_1.iloc[:, -7:].mean(axis=1)
)

forecast_7day_sample_window_1 = pd.DataFrame(
    np.repeat(
        avg_last_7_sample_window_1.to_numpy()[:, None],
        28,
        axis=1
    ),
    index=validation_sales_1000_window_1.index,
    columns=validation_sales_1000_window_1.columns
)

mae_7day_sample_window_1 = (
    validation_sales_1000_window_1 - forecast_7day_sample_window_1
).abs().to_numpy().mean()

mae_7day_sample_window_1

avg_last_28-days

In [ ]:
avg_last_28_sample_window_1 = (
    train_sales_1000_window_1.iloc[:, -28:].mean(axis=1)
)

forecast_28day_sample_window_1 = pd.DataFrame(
    np.repeat(
        avg_last_28_sample_window_1.to_numpy()[:, None],
        28,
        axis=1
    ),
    index=validation_sales_1000_window_1.index,
    columns=validation_sales_1000_window_1.columns
)

mae_28day_sample_window_1 = (
    validation_sales_1000_window_1 - forecast_28day_sample_window_1
).abs().to_numpy().mean()

mae_28day_sample_window_1

In [ ]:
comparison_window_1 = pd.DataFrame({
    "method": [
        "ML model",
        "7-day average",
        "28-day average"
    ],
    "MAE": [
        mae_ml_window_1,
        mae_7day_sample_window_1,
        mae_28day_sample_window_1
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_window_1

The Difference between the Actual_Sales and Predicted_Sales in Ml model.

In [ ]:
example_comparison = pd.DataFrame({
    "actual": validation_sales_1000_window_1.iloc[0],
    "predicted": forecast_ml_window_1.iloc[0]
})

example_comparison["absolute_error"] = (
    example_comparison["actual"] - example_comparison["predicted"]
).abs()

example_comparison.head(10)

train_sales_1000_window_2 from d_1830–d_1857.

In [ ]:
train_sales_1000_window_2 = daily_sales_1000.iloc[: , :-84]
validation_sales_1000_window_2 = daily_sales_1000.iloc[: , -84:-56]

train_sales_1000_window_2.shape , validation_sales_1000_window_2.shape

In [ ]:
sales_long_1000_window_2 = (
    train_sales_1000_window_2
    .rename_axis(index="series_id", columns="day")
    .stack()
    .rename("sales")
    .reset_index()
)

sales_long_1000_window_2["lag_1"] = (
    sales_long_1000_window_2.groupby("series_id")["sales"].shift(1)
)
sales_long_1000_window_2["lag_7"] = (
    sales_long_1000_window_2.groupby("series_id")["sales"].shift(7)
)
sales_long_1000_window_2["lag_28"] = (
    sales_long_1000_window_2.groupby("series_id")["sales"].shift(28)
)

model_data_1000_window_2 = sales_long_1000_window_2.dropna(
    subset=["lag_1", "lag_7", "lag_28"]
).copy()

X_train_1000_window_2 = model_data_1000_window_2[
    ["lag_1", "lag_7", "lag_28"]
]
y_train_1000_window_2 = model_data_1000_window_2["sales"]

X_train_1000_window_2.shape, y_train_1000_window_2.shape

In [ ]:
model_1000_window_2 = HistGradientBoostingRegressor(
    max_iter=50,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=50
)

model_1000_window_2.fit(
    X_train_1000_window_2,
    y_train_1000_window_2
)

In [ ]:
history_window_2 = train_sales_1000_window_2.to_numpy(dtype=float)
forecast_values_window_2 = np.zeros((history_window_2.shape[0], 28))

for day_step in range(28):
    next_features_window_2 = pd.DataFrame({
        "lag_1": history_window_2[:, -1],
        "lag_7": history_window_2[:, -7],
        "lag_28": history_window_2[:, -28],
    })

    next_prediction_window_2 = model_1000_window_2.predict(
        next_features_window_2
    )
    next_prediction_window_2 = np.maximum(next_prediction_window_2, 0)

    forecast_values_window_2[:, day_step] = next_prediction_window_2
    history_window_2 = np.column_stack(
        [history_window_2, next_prediction_window_2]
    )

forecast_ml_window_2 = pd.DataFrame(
    forecast_values_window_2,
    index=validation_sales_1000_window_2.index,
    columns=validation_sales_1000_window_2.columns
)

forecast_ml_window_2.iloc[:5, :5]

In [ ]:
error_ml_window_2 = (
    validation_sales_1000_window_2 - forecast_ml_window_2
).abs()

mae_ml_window_2 = error_ml_window_2.mean().mean()
mae_ml_window_2

7-day-average baseline on the same 1,000 series and window 2.

In [ ]:
avg_last_7_sample_window_2 = (
    train_sales_1000_window_2.iloc[:, -7:].mean(axis=1)
)

forecast_7day_sample_window_2 = pd.DataFrame(
    np.repeat(
        avg_last_7_sample_window_2.to_numpy()[:, None],
        28,
        axis=1
    ),
    index=validation_sales_1000_window_2.index,
    columns=validation_sales_1000_window_2.columns
)

mae_7day_sample_window_2 = (
    validation_sales_1000_window_2 - forecast_7day_sample_window_2
).abs().to_numpy().mean()

mae_7day_sample_window_2

28-day avg

In [ ]:
avg_last_28_sample_window_2 = (
    train_sales_1000_window_2.iloc[:, -28:].mean(axis=1)
)

forecast_28day_sample_window_2 = pd.DataFrame(
    np.repeat(
        avg_last_28_sample_window_2.to_numpy()[:, None],
        28,
        axis=1
    ),
    index=validation_sales_1000_window_2.index,
    columns=validation_sales_1000_window_2.columns
)

mae_28day_sample_window_2 = (
    validation_sales_1000_window_2 - forecast_28day_sample_window_2
).abs().to_numpy().mean()

mae_28day_sample_window_2

Final Comparison.

In [ ]:
comparison_window_2 = pd.DataFrame({
    "method": [
        "ML model",
        "7-day average",
        "28-day average"
    ],
    "MAE": [
        mae_ml_window_2,
        mae_7day_sample_window_2,
        mae_28day_sample_window_2
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_window_2

The latest 28-day test window on the same 1,000 series.

In [ ]:
train_sales_1000_latest = daily_sales_1000.iloc[:, :-28]
validation_sales_1000_latest = daily_sales_1000.iloc[:, -28:]

train_sales_1000_latest.shape, validation_sales_1000_latest.shape

In [ ]:
sales_long_1000_latest = (
    train_sales_1000_latest
    .rename_axis(index="series_id", columns="day")
    .stack()
    .rename("sales")
    .reset_index()
)

sales_long_1000_latest["lag_1"] = (
    sales_long_1000_latest.groupby("series_id")["sales"].shift(1)
)
sales_long_1000_latest["lag_7"] = (
    sales_long_1000_latest.groupby("series_id")["sales"].shift(7)
)
sales_long_1000_latest["lag_28"] = (
    sales_long_1000_latest.groupby("series_id")["sales"].shift(28)
)

model_data_1000_latest = sales_long_1000_latest.dropna(
    subset=["lag_1", "lag_7", "lag_28"]
).copy()

X_train_1000_latest = model_data_1000_latest[
    ["lag_1", "lag_7", "lag_28"]
]
y_train_1000_latest = model_data_1000_latest["sales"]

X_train_1000_latest.shape, y_train_1000_latest.shape

In [ ]:
model_1000_latest = HistGradientBoostingRegressor(
    max_iter=50,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=50
)

model_1000_latest.fit(
    X_train_1000_latest,
    y_train_1000_latest
)

In [ ]:
history_latest = train_sales_1000_latest.to_numpy(dtype=float)
forecast_values_latest = np.zeros((history_latest.shape[0], 28))

for day_step in range(28):
    next_features_latest = pd.DataFrame({
        "lag_1": history_latest[:, -1],
        "lag_7": history_latest[:, -7],
        "lag_28": history_latest[:, -28],
    })

    next_prediction_latest = model_1000_latest.predict(next_features_latest)
    next_prediction_latest = np.maximum(next_prediction_latest, 0)

    forecast_values_latest[:, day_step] = next_prediction_latest
    history_latest = np.column_stack([history_latest, next_prediction_latest])

forecast_ml_latest = pd.DataFrame(
    forecast_values_latest,
    index=validation_sales_1000_latest.index,
    columns=validation_sales_1000_latest.columns
)

forecast_ml_latest.iloc[:5, :5]

In [ ]:
error_ml_latest = (
    validation_sales_1000_latest - forecast_ml_latest
).abs()

mae_ml_latest = error_ml_latest.to_numpy().mean()
mae_ml_latest

for 7-day avg

In [ ]:
avg_last_7_sample_latest = (
    train_sales_1000_latest.iloc[:, -7:].mean(axis=1)
)

forecast_7day_sample_latest = pd.DataFrame(
    np.repeat(
        avg_last_7_sample_latest.to_numpy()[:, None],
        28,
        axis=1
    ),
    index=validation_sales_1000_latest.index,
    columns=validation_sales_1000_latest.columns
)

mae_7day_sample_latest = (
    validation_sales_1000_latest - forecast_7day_sample_latest
).abs().to_numpy().mean()

mae_7day_sample_latest

for 28-day avg

In [ ]:
avg_last_28_sample_latest = (
    train_sales_1000_latest.iloc[:, -28:].mean(axis=1)
)

forecast_28day_sample_latest = pd.DataFrame(
    np.repeat(
        avg_last_28_sample_latest.to_numpy()[:, None],
        28,
        axis=1
    ),
    index=validation_sales_1000_latest.index,
    columns=validation_sales_1000_latest.columns
)

mae_28day_sample_latest = (
    validation_sales_1000_latest - forecast_28day_sample_latest
).abs().to_numpy().mean()

mae_28day_sample_latest

Conclusion of latest-28 days...

In [ ]:
comparison_latest = pd.DataFrame({
    "method": [
        "ML model",
        "7-day average",
        "28-day average"
    ],
    "MAE": [
        mae_ml_latest,
        mae_7day_sample_latest,
        mae_28day_sample_latest
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_latest

All-Over Conclusion for ALL Modals

In [ ]:
summary_1000 = pd.DataFrame({
    "method": [
        "ML model",
        "7-day average",
        "28-day average"
    ],
    "MAE_window_1": [
        mae_ml_window_1,
        mae_7day_sample_window_1,
        mae_28day_sample_window_1
    ],
    "MAE_window_2": [
        mae_ml_window_2,
        mae_7day_sample_window_2,
        mae_28day_sample_window_2
    ],
    "MAE_latest": [
        mae_ml_latest,
        mae_7day_sample_latest,
        mae_28day_sample_latest
    ]
})

summary_1000["Mean_MAE"] = summary_1000[
    ["MAE_window_1", "MAE_window_2", "MAE_latest"]
].mean(axis=1)

summary_1000.sort_values("Mean_MAE").reset_index(drop=True)

### First ML experiment: 1,000 series

Across three 28-day test windows, the 28-day average had the lowest mean MAE (1.0381) on this 1,000-series sample. The 7-day average scored 1.0600, and the ML model scored 1.1226. The ML model did not beat either simple average in any of the three windows.

In the earlier evaluation using all 30,490 series, the 7-day average was slightly better than the 28-day average. The rankings differ on this 1,000-series sample, so keep the sample results separate from the full-dataset results.

_______________________________________________________________________________

### Second ML experiment: Add weekday information

**Question:** Does telling the model the target day's weekday improve its 28-day sales forecasts?

The first ML model used sales from 1, 7, and 28 days earlier. This version will use those same sales clues plus the target day's weekday from `calendar.csv`.

We will use the same 1,000 sampled series and the same test windows:

| Window | Training days | Validation days |
|---|---|---|
| Window 1 | d_1–d_1857 | d_1858–d_1885 |
| Window 2 | d_1–d_1829 | d_1830–d_1857 |
| Latest | d_1–d_1885 | d_1886–d_1913 |

For each window, train using only its training days, then compare the forecast MAE with the first ML model and the 7-day and 28-day averages. Lower MAE means smaller average errors. This test will show whether weekday information helps; it may or may not improve the model.

In [ ]:
calendar_weekdays = pd.read_csv(
    data_dir / "calendar.csv",
    usecols=["d", "weekday", "wday"]
)

calendar_weekdays.head()

In [ ]:
sales_long_1000_latest_weekday = sales_long_1000_latest.merge(
    calendar_weekdays[["d", "weekday", "wday"]],
    left_on="day",
    right_on="d",
    how="left"
).drop(columns="d")

sales_long_1000_latest_weekday[
    ["series_id", "day", "sales", "lag_1", "lag_7", "lag_28", "weekday", "wday"]
].head()

In [ ]:
model_data_1000_latest_weekday = sales_long_1000_latest_weekday.dropna(
    subset=["lag_1", "lag_7", "lag_28"]
).copy()

weekday_dummies_latest = pd.get_dummies(
    model_data_1000_latest_weekday["weekday"],
    prefix="weekday",
    dtype=int
)

model_data_1000_latest_weekday = pd.concat(
    [model_data_1000_latest_weekday, weekday_dummies_latest],
    axis=1
)

weekday_feature_columns_latest = weekday_dummies_latest.columns.tolist()
feature_columns_latest = (
    ["lag_1", "lag_7", "lag_28"] + weekday_feature_columns_latest
)

X_train_1000_latest_weekday = model_data_1000_latest_weekday[
    feature_columns_latest
]
y_train_1000_latest_weekday = model_data_1000_latest_weekday["sales"]

X_train_1000_latest_weekday.shape, y_train_1000_latest_weekday.shape

In [ ]:
model_1000_latest_weekday = HistGradientBoostingRegressor(
    max_iter=50,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=50
)

model_1000_latest_weekday.fit(
    X_train_1000_latest_weekday,
    y_train_1000_latest_weekday
)

In [ ]:
weekday_by_day = calendar_weekdays.set_index("d")["weekday"]

history_latest_weekday = train_sales_1000_latest.to_numpy(dtype=float)
forecast_values_latest_weekday = np.zeros(
    (history_latest_weekday.shape[0], 28)
)

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history_latest_weekday[:, -1],
        "lag_7": history_latest_weekday[:, -7],
        "lag_28": history_latest_weekday[:, -28],
    })

    target_day = validation_sales_1000_latest.columns[day_step]
    weekday_name = weekday_by_day.loc[target_day]

    for weekday_column in weekday_feature_columns_latest:
        next_features[weekday_column] = int(
            weekday_column == f"weekday_{weekday_name}"
        )

    next_features = next_features[feature_columns_latest]

    next_prediction = model_1000_latest_weekday.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values_latest_weekday[:, day_step] = next_prediction
    history_latest_weekday = np.column_stack(
        [history_latest_weekday, next_prediction]
    )

forecast_ml_weekday_latest = pd.DataFrame(
    forecast_values_latest_weekday,
    index=validation_sales_1000_latest.index,
    columns=validation_sales_1000_latest.columns
)

forecast_ml_weekday_latest.iloc[:5, :5]

In [ ]:
error_ml_weekday_latest = (
    validation_sales_1000_latest - forecast_ml_weekday_latest
).abs()

mae_ml_weekday_latest = error_ml_weekday_latest.to_numpy().mean()
mae_ml_weekday_latest

In [ ]:
comparison_latest_with_weekday = pd.DataFrame({
    "method": [
        "ML with weekday",
        "Original ML",
        "7-day average",
        "28-day average"
    ],
    "MAE": [
        mae_ml_weekday_latest,
        mae_ml_latest,
        mae_7day_sample_latest,
        mae_28day_sample_latest
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_latest_with_weekday

Testing Weekday_ML_Model for Window_1

In [ ]:
sales_long_1000_window_1_weekday = sales_long_1000.merge(
    calendar_weekdays[["d", "weekday"]],
    left_on="day",
    right_on="d",
    how="left"
).drop(columns="d")

model_data_1000_window_1_weekday = (
    sales_long_1000_window_1_weekday
    .dropna(subset=["lag_1", "lag_7", "lag_28"])
    .copy()
)

weekday_dummies_window_1 = pd.get_dummies(
    model_data_1000_window_1_weekday["weekday"],
    prefix="weekday",
    dtype=int
)

model_data_1000_window_1_weekday = pd.concat(
    [model_data_1000_window_1_weekday, weekday_dummies_window_1],
    axis=1
)

weekday_feature_columns_window_1 = weekday_dummies_window_1.columns.tolist()
feature_columns_window_1 = (
    ["lag_1", "lag_7", "lag_28"] + weekday_feature_columns_window_1
)

X_train_1000_window_1_weekday = model_data_1000_window_1_weekday[
    feature_columns_window_1
]
y_train_1000_window_1_weekday = model_data_1000_window_1_weekday["sales"]

X_train_1000_window_1_weekday.shape, y_train_1000_window_1_weekday.shape

In [ ]:
model_1000_window_1_weekday = HistGradientBoostingRegressor(
    max_iter=50,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=50
)

model_1000_window_1_weekday.fit(
    X_train_1000_window_1_weekday,
    y_train_1000_window_1_weekday
)

In [ ]:
history_window_1_weekday = train_sales_1000_window_1.to_numpy(dtype=float)
forecast_values_window_1_weekday = np.zeros(
    (history_window_1_weekday.shape[0], 28)
)

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history_window_1_weekday[:, -1],
        "lag_7": history_window_1_weekday[:, -7],
        "lag_28": history_window_1_weekday[:, -28],
    })

    target_day = validation_sales_1000_window_1.columns[day_step]
    weekday_name = weekday_by_day.loc[target_day]

    for weekday_column in weekday_feature_columns_window_1:
        next_features[weekday_column] = int(
            weekday_column == f"weekday_{weekday_name}"
        )

    next_features = next_features[feature_columns_window_1]

    next_prediction = model_1000_window_1_weekday.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values_window_1_weekday[:, day_step] = next_prediction
    history_window_1_weekday = np.column_stack(
        [history_window_1_weekday, next_prediction]
    )

forecast_ml_weekday_window_1 = pd.DataFrame(
    forecast_values_window_1_weekday,
    index=validation_sales_1000_window_1.index,
    columns=validation_sales_1000_window_1.columns
)

forecast_ml_weekday_window_1.iloc[:5, :5]

In [ ]:
error_ml_weekday_window_1 = (
    validation_sales_1000_window_1 - forecast_ml_weekday_window_1
).abs()

mae_ml_weekday_window_1 = error_ml_weekday_window_1.to_numpy().mean()
mae_ml_weekday_window_1

Testing Weekday_ML_Model for Window_2

In [ ]:
sales_long_1000_window_2_weekday = sales_long_1000_window_2.merge(
    calendar_weekdays[["d", "weekday"]],
    left_on="day",
    right_on="d",
    how="left"
).drop(columns="d")

model_data_1000_window_2_weekday = (
    sales_long_1000_window_2_weekday
    .dropna(subset=["lag_1", "lag_7", "lag_28"])
    .copy()
)

weekday_dummies_window_2 = pd.get_dummies(
    model_data_1000_window_2_weekday["weekday"],
    prefix="weekday",
    dtype=int
)

model_data_1000_window_2_weekday = pd.concat(
    [model_data_1000_window_2_weekday, weekday_dummies_window_2],
    axis=1
)

weekday_feature_columns_window_2 = weekday_dummies_window_2.columns.tolist()
feature_columns_window_2 = (
    ["lag_1", "lag_7", "lag_28"] + weekday_feature_columns_window_2
)

X_train_1000_window_2_weekday = model_data_1000_window_2_weekday[
    feature_columns_window_2
]
y_train_1000_window_2_weekday = model_data_1000_window_2_weekday["sales"]

X_train_1000_window_2_weekday.shape, y_train_1000_window_2_weekday.shape

In [ ]:
model_1000_window_2_weekday = HistGradientBoostingRegressor(
    max_iter=50,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=50
)

model_1000_window_2_weekday.fit(
    X_train_1000_window_2_weekday,
    y_train_1000_window_2_weekday
)

In [ ]:
history_window_2_weekday = train_sales_1000_window_2.to_numpy(dtype=float)
forecast_values_window_2_weekday = np.zeros(
    (history_window_2_weekday.shape[0], 28)
)

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history_window_2_weekday[:, -1],
        "lag_7": history_window_2_weekday[:, -7],
        "lag_28": history_window_2_weekday[:, -28],
    })

    target_day = validation_sales_1000_window_2.columns[day_step]
    weekday_name = weekday_by_day.loc[target_day]

    for weekday_column in weekday_feature_columns_window_2:
        next_features[weekday_column] = int(
            weekday_column == f"weekday_{weekday_name}"
        )

    next_features = next_features[feature_columns_window_2]

    next_prediction = model_1000_window_2_weekday.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values_window_2_weekday[:, day_step] = next_prediction
    history_window_2_weekday = np.column_stack(
        [history_window_2_weekday, next_prediction]
    )

forecast_ml_weekday_window_2 = pd.DataFrame(
    forecast_values_window_2_weekday,
    index=validation_sales_1000_window_2.index,
    columns=validation_sales_1000_window_2.columns
)

forecast_ml_weekday_window_2.iloc[:5, :5]

In [ ]:
error_ml_weekday_window_2 = (
    validation_sales_1000_window_2 - forecast_ml_weekday_window_2
).abs()

mae_ml_weekday_window_2 = error_ml_weekday_window_2.to_numpy().mean()
mae_ml_weekday_window_2

In [ ]:
comparison_window_2_with_weekday = pd.DataFrame({
    "method": [
        "Original ML",
        "ML with weekday",
        "7-day average",
        "28-day average"
    ],
    "MAE": [
        mae_ml_window_2,
        mae_ml_weekday_window_2,
        mae_7day_sample_window_2,
        mae_28day_sample_window_2
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_window_2_with_weekday

In [ ]:
comparison_window_1_with_weekday = pd.DataFrame({
    "method": [
        "Original ML",
        "ML with weekday",
        "7-day average",
        "28-day average"
    ],
    "MAE": [
        mae_ml_window_1,
        mae_ml_weekday_window_1,
        mae_7day_sample_window_1,
        mae_28day_sample_window_1
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_window_1_with_weekday

In [ ]:
summary_weekday_1000 = pd.DataFrame({
    "method": [
        "Original ML",
        "ML with weekday",
        "7-day average",
        "28-day average"
    ],
    "MAE_window_1": [
        mae_ml_window_1,
        mae_ml_weekday_window_1,
        mae_7day_sample_window_1,
        mae_28day_sample_window_1
    ],
    "MAE_window_2": [
        mae_ml_window_2,
        mae_ml_weekday_window_2,
        mae_7day_sample_window_2,
        mae_28day_sample_window_2
    ],
    "MAE_latest": [
        mae_ml_latest,
        mae_ml_weekday_latest,
        mae_7day_sample_latest,
        mae_28day_sample_latest
    ]
})

summary_weekday_1000["Mean_MAE"] = summary_weekday_1000[
    ["MAE_window_1", "MAE_window_2", "MAE_latest"]
].mean(axis=1)

summary_weekday_1000.sort_values("Mean_MAE").reset_index(drop=True)

### Third ML experiment: Add product and store information

**Question:** Does product and store information improve the weekday-aware ML model?

In the previous test, adding weekday information did not improve the model overall: its mean MAE was 1.1237, compared with 1.1226 for the original ML model. On this 1,000-series sample, the 28-day average still performed best, with a mean MAE of 1.0381.

For this experiment, we will keep the sales-lag and weekday clues, and add product/store metadata from the sales data: `item_id`, `dept_id`, `cat_id`, `store_id`, and `state_id`. We will encode these labels in a way the model can use.

We will use the same 1,000 series and the same three 28-day test windows. For each window, the model will train only on earlier days. We will compare its MAE with the previous ML models and the 7-day and 28-day averages. Lower MAE means smaller average errors.

In [ ]:
sample_1000[
    ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
].nunique()

In [ ]:
series_metadata_1000 = (
    sample_1000[["item_id", "dept_id", "cat_id", "store_id", "state_id"]]
    .rename_axis("series_id")
    .reset_index()
)

sales_long_1000_latest_metadata = sales_long_1000_latest_weekday.merge(
    series_metadata_1000,
    on="series_id",
    how="left"
)

sales_long_1000_latest_metadata[
    ["series_id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "day"]
].head()

In [ ]:
model_data_1000_latest_productstore = (
    sales_long_1000_latest_metadata
    .dropna(subset=["lag_1", "lag_7", "lag_28"])
    .copy()
)

weekday_dummies_productstore = pd.get_dummies(
    model_data_1000_latest_productstore["weekday"],
    prefix="weekday",
    dtype=int
)

model_data_1000_latest_productstore = pd.concat(
    [model_data_1000_latest_productstore, weekday_dummies_productstore],
    axis=1
)

metadata_category_columns = [
    "dept_id", "cat_id", "store_id", "state_id"
]

for column in metadata_category_columns:
    model_data_1000_latest_productstore[column] = (
        model_data_1000_latest_productstore[column].astype("category")
    )

feature_columns_productstore = (
    ["lag_1", "lag_7", "lag_28"]
    + weekday_dummies_productstore.columns.tolist()
    + metadata_category_columns
)

X_train_1000_latest_productstore = model_data_1000_latest_productstore[
    feature_columns_productstore
]
y_train_1000_latest_productstore = model_data_1000_latest_productstore["sales"]

X_train_1000_latest_productstore.shape, y_train_1000_latest_productstore.shape

In [ ]:
model_1000_latest_productstore = HistGradientBoostingRegressor(
    max_iter=50,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=50,
    categorical_features="from_dtype"
)

model_1000_latest_productstore.fit(
    X_train_1000_latest_productstore,
    y_train_1000_latest_productstore
)

In [ ]:
weekday_by_day = calendar_weekdays.set_index("d")["weekday"]

metadata_for_forecast = (
    sample_1000[metadata_category_columns]
    .reset_index(drop=True)
    .copy()
)

for column in metadata_category_columns:
    metadata_for_forecast[column] = metadata_for_forecast[column].astype("category")

history_latest_productstore = train_sales_1000_latest.to_numpy(dtype=float)
forecast_values_latest_productstore = np.zeros(
    (history_latest_productstore.shape[0], 28)
)

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history_latest_productstore[:, -1],
        "lag_7": history_latest_productstore[:, -7],
        "lag_28": history_latest_productstore[:, -28],
    })

    target_day = validation_sales_1000_latest.columns[day_step]
    weekday_name = weekday_by_day.loc[target_day]

    for weekday_column in weekday_dummies_productstore.columns:
        next_features[weekday_column] = int(
            weekday_column == f"weekday_{weekday_name}"
        )

    for column in metadata_category_columns:
        next_features[column] = metadata_for_forecast[column]

    next_features = next_features[feature_columns_productstore]

    next_prediction = model_1000_latest_productstore.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values_latest_productstore[:, day_step] = next_prediction
    history_latest_productstore = np.column_stack(
        [history_latest_productstore, next_prediction]
    )

forecast_ml_productstore_latest = pd.DataFrame(
    forecast_values_latest_productstore,
    index=validation_sales_1000_latest.index,
    columns=validation_sales_1000_latest.columns
)

forecast_ml_productstore_latest.iloc[:5, :5]

In [ ]:
error_ml_productstore_latest = (
    validation_sales_1000_latest - forecast_ml_productstore_latest
).abs()

mae_ml_productstore_latest = error_ml_productstore_latest.to_numpy().mean()
mae_ml_productstore_latest

In [ ]:
comparison_latest_all_models = pd.DataFrame({
    "method": [
        "28-day average",
        "7-day average",
        "Original ML",
        "ML with weekday",
        "ML with weekday and product/store"
    ],
    "MAE": [
        mae_28day_sample_latest,
        mae_7day_sample_latest,
        mae_ml_latest,
        mae_ml_weekday_latest,
        mae_ml_productstore_latest
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_latest_all_models

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

metadata_category_columns = ["dept_id", "cat_id", "store_id", "state_id"]
weekday_by_day = calendar_weekdays.set_index("d")["weekday"]

def fit_forecast_productstore(sales_long, train_sales, validation_sales):
    # Add weekday and series metadata to each training-day row
    model_data = sales_long.merge(
        calendar_weekdays[["d", "weekday"]],
        left_on="day",
        right_on="d",
        how="left"
    ).drop(columns="d")

    model_data = model_data.merge(
        series_metadata_1000,
        on="series_id",
        how="left"
    )

    # Remove the first 28 days, which don't have all three lag values
    model_data = model_data.dropna(
        subset=["lag_1", "lag_7", "lag_28"]
    ).copy()

    # Turn weekday names into seven numeric yes/no clues
    weekday_dummies = pd.get_dummies(
        model_data["weekday"],
        prefix="weekday",
        dtype=int
    )
    model_data = pd.concat([model_data, weekday_dummies], axis=1)
    weekday_columns = weekday_dummies.columns.tolist()

    # Mark the low-cardinality metadata as categories
    for column in metadata_category_columns:
        model_data[column] = model_data[column].astype("category")

    category_levels = {
        column: model_data[column].cat.categories
        for column in metadata_category_columns
    }

    feature_columns = (
        ["lag_1", "lag_7", "lag_28"]
        + weekday_columns
        + metadata_category_columns
    )
    X_train = model_data[feature_columns]
    y_train = model_data["sales"]

    # Train a fresh model for this window
    model = HistGradientBoostingRegressor(
        max_iter=50,
        max_leaf_nodes=31,
        learning_rate=0.1,
        random_state=50,
        categorical_features="from_dtype"
    )
    model.fit(X_train, y_train)

    # Keep each series' metadata in the same row order as the sales matrix
    metadata_for_forecast = (
        series_metadata_1000[metadata_category_columns]
        .reset_index(drop=True)
        .copy()
    )
    for column in metadata_category_columns:
        metadata_for_forecast[column] = pd.Categorical(
            metadata_for_forecast[column],
            categories=category_levels[column]
        )

    # Forecast the 28 days one at a time
    history = train_sales.to_numpy(dtype=float)
    forecast_values = np.zeros((history.shape[0], 28))

    for day_step in range(28):
        next_features = pd.DataFrame({
            "lag_1": history[:, -1],
            "lag_7": history[:, -7],
            "lag_28": history[:, -28],
        })

        target_day = validation_sales.columns[day_step]
        weekday_name = weekday_by_day.loc[target_day]

        for column in weekday_columns:
            next_features[column] = int(
                column == f"weekday_{weekday_name}"
            )

        for column in metadata_category_columns:
            next_features[column] = metadata_for_forecast[column]

        next_features = next_features[feature_columns]
        next_prediction = model.predict(next_features)
        next_prediction = np.maximum(next_prediction, 0)

        forecast_values[:, day_step] = next_prediction
        history = np.column_stack([history, next_prediction])

    forecast = pd.DataFrame(
        forecast_values,
        index=validation_sales.index,
        columns=validation_sales.columns
    )

    mae = np.abs(
        validation_sales.to_numpy(dtype=float) - forecast_values
    ).mean()

    return model, forecast, mae


model_1000_window_1_productstore, forecast_ml_productstore_window_1, mae_ml_productstore_window_1 = fit_forecast_productstore(
    sales_long_1000,
    train_sales_1000_window_1,
    validation_sales_1000_window_1
)

model_1000_window_2_productstore, forecast_ml_productstore_window_2, mae_ml_productstore_window_2 = fit_forecast_productstore(
    sales_long_1000_window_2,
    train_sales_1000_window_2,
    validation_sales_1000_window_2
)

print("Window 1 MAE:", mae_ml_productstore_window_1)
print("Window 2 MAE:", mae_ml_productstore_window_2)

In [ ]:
summary_all_1000 = pd.DataFrame({
    "method": [
        "28-day average",
        "7-day average",
        "Original ML",
        "ML with weekday",
        "ML with weekday and product/store"
    ],
    "MAE_window_1": [
        mae_28day_sample_window_1,
        mae_7day_sample_window_1,
        mae_ml_window_1,
        mae_ml_weekday_window_1,
        mae_ml_productstore_window_1
    ],
    "MAE_window_2": [
        mae_28day_sample_window_2,
        mae_7day_sample_window_2,
        mae_ml_window_2,
        mae_ml_weekday_window_2,
        mae_ml_productstore_window_2
    ],
    "MAE_latest": [
        mae_28day_sample_latest,
        mae_7day_sample_latest,
        mae_ml_latest,
        mae_ml_weekday_latest,
        mae_ml_productstore_latest
    ]
})

summary_all_1000["Mean_MAE"] = summary_all_1000[
    ["MAE_window_1", "MAE_window_2", "MAE_latest"]
].mean(axis=1)

summary_all_1000.sort_values("Mean_MAE").reset_index(drop=True)

In [ ]:
import importlib.util

importlib.util.find_spec("catboost") is not None

In [ ]:
import catboost
catboost.__version__

Using CATBoost in ML Modal.

In [ ]:
catboost_categorical_columns = [
    "weekday", "item_id", "dept_id", "cat_id", "store_id", "state_id"
]

sales_long_1000_latest_catboost = (
    sales_long_1000_latest
    .merge(
        calendar_weekdays[["d", "weekday"]],
        left_on="day",
        right_on="d",
        how="left"
    )
    .drop(columns="d")
    .merge(series_metadata_1000, on="series_id", how="left")
)

model_data_1000_latest_catboost = sales_long_1000_latest_catboost.dropna(
    subset=["lag_1", "lag_7", "lag_28"]
).copy()

catboost_feature_columns = (
    ["lag_1", "lag_7", "lag_28"] + catboost_categorical_columns
)

X_train_1000_latest_catboost = model_data_1000_latest_catboost[
    catboost_feature_columns
].copy()

for column in catboost_categorical_columns:
    X_train_1000_latest_catboost[column] = (
        X_train_1000_latest_catboost[column].astype(str)
    )

y_train_1000_latest_catboost = model_data_1000_latest_catboost["sales"]

X_train_1000_latest_catboost.shape, y_train_1000_latest_catboost.shape

In [ ]:
del X_train_1000_latest_catboost, y_train_1000_latest_catboost

import gc
gc.collect()

catboost_training_sample = (
    model_data_1000_latest_catboost
    .groupby("series_id", group_keys=False)
    .sample(n=200, random_state=50)
)

X_train_1000_latest_catboost_small = catboost_training_sample[
    catboost_feature_columns
].copy()

for column in catboost_categorical_columns:
    X_train_1000_latest_catboost_small[column] = (
        X_train_1000_latest_catboost_small[column].astype(str)
    )

y_train_1000_latest_catboost_small = catboost_training_sample["sales"]

X_train_1000_latest_catboost_small.shape, y_train_1000_latest_catboost_small.shape

In [ ]:
model_1000_latest_catboost = catboost.CatBoostRegressor(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    loss_function="MAE",
    random_seed=50,
    verbose=25,
    thread_count=2,
    allow_writing_files=False
)

model_1000_latest_catboost.fit(
    X_train_1000_latest_catboost_small,
    y_train_1000_latest_catboost_small,
    cat_features=catboost_categorical_columns
)

In [ ]:
weekday_by_day = calendar_weekdays.set_index("d")["weekday"]

metadata_columns = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id"
]

metadata_for_forecast = (
    series_metadata_1000[metadata_columns]
    .reset_index(drop=True)
    .astype(str)
)

history_catboost_latest = train_sales_1000_latest.to_numpy(dtype=float)
forecast_values_catboost_latest = np.zeros(
    (history_catboost_latest.shape[0], 28)
)

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history_catboost_latest[:, -1],
        "lag_7": history_catboost_latest[:, -7],
        "lag_28": history_catboost_latest[:, -28],
    })

    target_day = validation_sales_1000_latest.columns[day_step]
    next_features["weekday"] = str(weekday_by_day.loc[target_day])

    for column in metadata_columns:
        next_features[column] = metadata_for_forecast[column]

    next_features = next_features[catboost_feature_columns]

    next_prediction = model_1000_latest_catboost.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values_catboost_latest[:, day_step] = next_prediction
    history_catboost_latest = np.column_stack(
        [history_catboost_latest, next_prediction]
    )

forecast_catboost_latest = pd.DataFrame(
    forecast_values_catboost_latest,
    index=validation_sales_1000_latest.index,
    columns=validation_sales_1000_latest.columns
)

forecast_catboost_latest.iloc[:5, :5]

In [ ]:
error_catboost_latest = (
    validation_sales_1000_latest - forecast_catboost_latest
).abs()

mae_catboost_latest = error_catboost_latest.to_numpy().mean()
mae_catboost_latest

In [ ]:
comparison_latest_with_catboost = pd.DataFrame({
    "method": [
        "28-day average",
        "7-day average",
        "Original ML",
        "ML with weekday",
        "ML with weekday and product/store",
        "CatBoost with item_id"
    ],
    "MAE": [
        mae_28day_sample_latest,
        mae_7day_sample_latest,
        mae_ml_latest,
        mae_ml_weekday_latest,
        mae_ml_productstore_latest,
        mae_catboost_latest
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_latest_with_catboost

CatBoost for window_1

In [ ]:
sales_long_1000_window_1_catboost = (
    sales_long_1000
    .merge(
        calendar_weekdays[["d", "weekday"]],
        left_on="day",
        right_on="d",
        how="left"
    )
    .drop(columns="d")
    .merge(series_metadata_1000, on="series_id", how="left")
)

model_data_1000_window_1_catboost = (
    sales_long_1000_window_1_catboost
    .dropna(subset=["lag_1", "lag_7", "lag_28"])
    .copy()
)

catboost_sample_window_1 = (
    model_data_1000_window_1_catboost
    .groupby("series_id", group_keys=False)
    .sample(n=200, random_state=50)
)

X_train_1000_window_1_catboost_small = catboost_sample_window_1[
    catboost_feature_columns
].copy()

for column in catboost_categorical_columns:
    X_train_1000_window_1_catboost_small[column] = (
        X_train_1000_window_1_catboost_small[column].astype(str)
    )

y_train_1000_window_1_catboost_small = catboost_sample_window_1["sales"]

X_train_1000_window_1_catboost_small.shape, y_train_1000_window_1_catboost_small.shape

In [ ]:
import gc

del sales_long_1000_window_1_catboost
del model_data_1000_window_1_catboost
del catboost_sample_window_1
gc.collect()

model_1000_window_1_catboost = catboost.CatBoostRegressor(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    loss_function="MAE",
    random_seed=50,
    verbose=25,
    thread_count=2,
    allow_writing_files=False
)

model_1000_window_1_catboost.fit(
    X_train_1000_window_1_catboost_small,
    y_train_1000_window_1_catboost_small,
    cat_features=catboost_categorical_columns
)

In [ ]:
weekday_by_day = calendar_weekdays.set_index("d")["weekday"]

catboost_metadata_columns = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id"
]

metadata_for_forecast_window_1 = (
    series_metadata_1000[catboost_metadata_columns]
    .reset_index(drop=True)
    .astype(str)
)

history_catboost_window_1 = train_sales_1000_window_1.to_numpy(dtype=float)
forecast_values_catboost_window_1 = np.zeros(
    (history_catboost_window_1.shape[0], 28)
)

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history_catboost_window_1[:, -1],
        "lag_7": history_catboost_window_1[:, -7],
        "lag_28": history_catboost_window_1[:, -28],
    })

    target_day = validation_sales_1000_window_1.columns[day_step]
    next_features["weekday"] = str(weekday_by_day.loc[target_day])

    for column in catboost_metadata_columns:
        next_features[column] = metadata_for_forecast_window_1[column]

    next_features = next_features[catboost_feature_columns]

    next_prediction = model_1000_window_1_catboost.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values_catboost_window_1[:, day_step] = next_prediction
    history_catboost_window_1 = np.column_stack(
        [history_catboost_window_1, next_prediction]
    )

forecast_catboost_window_1 = pd.DataFrame(
    forecast_values_catboost_window_1,
    index=validation_sales_1000_window_1.index,
    columns=validation_sales_1000_window_1.columns
)

forecast_catboost_window_1.iloc[:5, :5]

In [ ]:
error_catboost_window_1 = (
    validation_sales_1000_window_1 - forecast_catboost_window_1
).abs()

mae_catboost_window_1 = error_catboost_window_1.to_numpy().mean()
mae_catboost_window_1

Using CatBoost for window_2

In [ ]:
sales_long_1000_window_2_catboost = (
    sales_long_1000_window_2
    .merge(
        calendar_weekdays[["d", "weekday"]],
        left_on="day",
        right_on="d",
        how="left"
    )
    .drop(columns="d")
    .merge(series_metadata_1000, on="series_id", how="left")
)

model_data_1000_window_2_catboost = (
    sales_long_1000_window_2_catboost
    .dropna(subset=["lag_1", "lag_7", "lag_28"])
    .copy()
)

catboost_sample_window_2 = (
    model_data_1000_window_2_catboost
    .groupby("series_id", group_keys=False)
    .sample(n=200, random_state=50)
)

X_train_1000_window_2_catboost_small = catboost_sample_window_2[
    catboost_feature_columns
].copy()

for column in catboost_categorical_columns:
    X_train_1000_window_2_catboost_small[column] = (
        X_train_1000_window_2_catboost_small[column].astype(str)
    )

y_train_1000_window_2_catboost_small = catboost_sample_window_2["sales"]

X_train_1000_window_2_catboost_small.shape, y_train_1000_window_2_catboost_small.shape

In [ ]:
import gc

del sales_long_1000_window_2_catboost
del model_data_1000_window_2_catboost
del catboost_sample_window_2
gc.collect()

model_1000_window_2_catboost = catboost.CatBoostRegressor(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    loss_function="MAE",
    random_seed=50,
    verbose=25,
    thread_count=2,
    allow_writing_files=False
)

model_1000_window_2_catboost.fit(
    X_train_1000_window_2_catboost_small,
    y_train_1000_window_2_catboost_small,
    cat_features=catboost_categorical_columns
)

In [ ]:
metadata_for_forecast_window_2 = (
    series_metadata_1000[catboost_metadata_columns]
    .reset_index(drop=True)
    .astype(str)
)

history_catboost_window_2 = train_sales_1000_window_2.to_numpy(dtype=float)
forecast_values_catboost_window_2 = np.zeros(
    (history_catboost_window_2.shape[0], 28)
)

for day_step in range(28):
    next_features = pd.DataFrame({
        "lag_1": history_catboost_window_2[:, -1],
        "lag_7": history_catboost_window_2[:, -7],
        "lag_28": history_catboost_window_2[:, -28],
    })

    target_day = validation_sales_1000_window_2.columns[day_step]
    next_features["weekday"] = str(weekday_by_day.loc[target_day])

    for column in catboost_metadata_columns:
        next_features[column] = metadata_for_forecast_window_2[column]

    next_features = next_features[catboost_feature_columns]

    next_prediction = model_1000_window_2_catboost.predict(next_features)
    next_prediction = np.maximum(next_prediction, 0)

    forecast_values_catboost_window_2[:, day_step] = next_prediction
    history_catboost_window_2 = np.column_stack(
        [history_catboost_window_2, next_prediction]
    )

forecast_catboost_window_2 = pd.DataFrame(
    forecast_values_catboost_window_2,
    index=validation_sales_1000_window_2.index,
    columns=validation_sales_1000_window_2.columns
)

forecast_catboost_window_2.iloc[:5, :5]

In [ ]:
error_catboost_window_2 = (
    validation_sales_1000_window_2 - forecast_catboost_window_2
).abs()

mae_catboost_window_2 = error_catboost_window_2.to_numpy().mean()
mae_catboost_window_2

In [ ]:
summary_all_with_catboost = pd.DataFrame({
    "method": [
        "28-day average",
        "7-day average",
        "Original ML",
        "ML with weekday",
        "ML with weekday and product/store",
        "CatBoost with item_id"
    ],
    "MAE_window_1": [
        mae_28day_sample_window_1,
        mae_7day_sample_window_1,
        mae_ml_window_1,
        mae_ml_weekday_window_1,
        mae_ml_productstore_window_1,
        mae_catboost_window_1
    ],
    "MAE_window_2": [
        mae_28day_sample_window_2,
        mae_7day_sample_window_2,
        mae_ml_window_2,
        mae_ml_weekday_window_2,
        mae_ml_productstore_window_2,
        mae_catboost_window_2
    ],
    "MAE_latest": [
        mae_28day_sample_latest,
        mae_7day_sample_latest,
        mae_ml_latest,
        mae_ml_weekday_latest,
        mae_ml_productstore_latest,
        mae_catboost_latest
    ]
})

summary_all_with_catboost["Mean_MAE"] = summary_all_with_catboost[
    ["MAE_window_1", "MAE_window_2", "MAE_latest"]
].mean(axis=1)

summary_all_with_catboost.sort_values("Mean_MAE").reset_index(drop=True)

In [ ]:
all_sales.shape, daily_sales_all.shape, calendar_weekdays.shape, catboost.__version__

## CatBoost pilot: product and store information

For this pilot, we’ll train CatBoost using five sampled historical days from each of the 30,490 product-store series. For each sampled day, the model will use sales from 1, 7, and 28 days earlier, along with the weekday and product/store identifiers. The actual sales on the sampled day are the target it learns to predict.

This creates about 152,450 training examples instead of turning all historical sales into tens of millions of rows, which helps limit memory use.

We’ll use this model to forecast the latest 28 days for all 30,490 series, then compare its MAE with the 7-day average on the same test window. This is a pilot to check performance and computer resources; it is not yet a final model decision.

In [ ]:
import numpy as np
import pandas as pd

# For the latest test window, train through d_1885
train_sales_catboost_all = daily_sales_all.iloc[:, :-28]
sales_matrix = train_sales_catboost_all.to_numpy(copy=False)

n_series, n_train_days = sales_matrix.shape
samples_per_series = 5

# Pick five historical target days for each series
rng = np.random.default_rng(50)
series_rows = np.repeat(np.arange(n_series), samples_per_series)
target_day_positions = rng.integers(
    low=28,
    high=n_train_days,
    size=n_series * samples_per_series
)

# Build the three lag features
X_train_catboost_all = pd.DataFrame({
    "lag_1": sales_matrix[series_rows, target_day_positions - 1],
    "lag_7": sales_matrix[series_rows, target_day_positions - 7],
    "lag_28": sales_matrix[series_rows, target_day_positions - 28],
})

# Add the weekday for each sampled target day
weekday_lookup = calendar_weekdays.set_index("d")["weekday"]
target_day_names = train_sales_catboost_all.columns.to_numpy()[target_day_positions]
X_train_catboost_all["weekday"] = (
    weekday_lookup.reindex(target_day_names).to_numpy()
)

# Add product and store identifiers as CatBoost categories
metadata_columns = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
metadata_values = all_sales[metadata_columns].astype(str).to_numpy()

for column_number, column_name in enumerate(metadata_columns):
    X_train_catboost_all[column_name] = metadata_values[
        series_rows, column_number
    ]

# The value on each sampled target day is what CatBoost will learn to predict
y_train_catboost_all = sales_matrix[
    series_rows, target_day_positions
]

catboost_categorical_columns = [
    "weekday", "item_id", "dept_id", "cat_id", "store_id", "state_id"
]
catboost_feature_columns = (
    ["lag_1", "lag_7", "lag_28"] + catboost_categorical_columns
)
X_train_catboost_all = X_train_catboost_all[catboost_feature_columns]

X_train_catboost_all.shape, y_train_catboost_all.shape

### Train the CatBoost pilot model

Now we’ll train one CatBoost model on the sampled examples. It will learn from the three sales lags plus weekday, product, and store information. We’re using 100 iterations and two CPU threads for this first pilot.

In [ ]:
model_catboost_all_latest = catboost.CatBoostRegressor(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    loss_function="MAE",
    random_seed=50,
    verbose=25,
    thread_count=2,
    allow_writing_files=False
)

model_catboost_all_latest.fit(
    X_train_catboost_all,
    y_train_catboost_all,
    cat_features=catboost_categorical_columns
)

In [ ]:
train_values = train_sales_catboost_all.to_numpy(copy=False)

validation_sales_catboost_all_latest = daily_sales_all.iloc[:, -28:]
n_series = train_values.shape[0]
n_forecast_days = validation_sales_catboost_all_latest.shape[1]

# Metadata and calendar values for the forecast rows
metadata_values = all_sales[metadata_columns].astype(str).to_numpy()
weekday_lookup = calendar_weekdays.set_index("d")["weekday"]

# Store one forecast value per series per day
forecast_values = np.zeros(
    (n_series, n_forecast_days),
    dtype=np.float32
)

def get_lag_values(lag, step):
    previous_forecast_step = step - lag

    if previous_forecast_step < 0:
        return train_values[:, previous_forecast_step]

    return forecast_values[:, previous_forecast_step]


for step, day in enumerate(validation_sales_catboost_all_latest.columns):
    future_features = pd.DataFrame({
        "lag_1": get_lag_values(1, step),
        "lag_7": get_lag_values(7, step),
        "lag_28": get_lag_values(28, step),
        "weekday": np.repeat(weekday_lookup.loc[day], n_series)
    })

    for column_number, column_name in enumerate(metadata_columns):
        future_features[column_name] = metadata_values[:, column_number]

    # Keep the feature order the model was trained with
    future_features = future_features[catboost_feature_columns]

    predicted_sales = model_catboost_all_latest.predict(future_features)

    # Sales cannot be negative
    forecast_values[:, step] = np.maximum(predicted_sales, 0)

    if (step + 1) % 7 == 0:
        print(f"Forecasted {step + 1} of {n_forecast_days} days")


forecast_catboost_all_latest = pd.DataFrame(
    forecast_values,
    index=validation_sales_catboost_all_latest.index,
    columns=validation_sales_catboost_all_latest.columns
)

forecast_catboost_all_latest.shape, forecast_catboost_all_latest.iloc[:5, :5]

In [ ]:
# Make the 7-day-average forecast for the same 28-day test window
avg_last_7_catboost_all = train_sales_catboost_all.iloc[:, -7:].mean(axis=1)

forecast_7day_catboost_all_latest = pd.DataFrame(
    np.repeat(
        avg_last_7_catboost_all.to_numpy()[:, None],
        n_forecast_days,
        axis=1
    ),
    index=validation_sales_catboost_all_latest.index,
    columns=validation_sales_catboost_all_latest.columns
)

# Calculate MAE for both methods on the same actual sales
mae_catboost_all_latest = (
    validation_sales_catboost_all_latest.to_numpy()
    - forecast_catboost_all_latest.to_numpy()
).__abs__().mean()

mae_7day_all_latest = (
    validation_sales_catboost_all_latest.to_numpy()
    - forecast_7day_catboost_all_latest.to_numpy()
).__abs__().mean()

comparison_latest_all = pd.DataFrame({
    "method": [
        "CatBoost with product/store info",
        "7-day average"
    ],
    "MAE": [
        mae_catboost_all_latest,
        mae_7day_all_latest
    ]
}).sort_values("MAE").reset_index(drop=True)

comparison_latest_all

### Test CatBoost on window 1

For this window, training uses sales through d_1857, and validation uses d_1858–d_1885. We’ll train a separate model using five sampled historical days per series, so it does not learn from dates after this test window.

In [ ]:
# Window 1: train through d_1857; validate on the next 28 days
train_sales_catboost_all_window_1 = daily_sales_all.iloc[:, :-56]
sales_matrix_window_1 = train_sales_catboost_all_window_1.to_numpy(copy=False)

n_series_window_1, n_train_days_window_1 = sales_matrix_window_1.shape
samples_per_series = 5

rng = np.random.default_rng(50)
series_rows_window_1 = np.repeat(
    np.arange(n_series_window_1),
    samples_per_series
)
target_positions_window_1 = rng.integers(
    low=28,
    high=n_train_days_window_1,
    size=n_series_window_1 * samples_per_series
)

# Create lag features for the sampled historical days
X_train_catboost_all_window_1 = pd.DataFrame({
    "lag_1": sales_matrix_window_1[
        series_rows_window_1, target_positions_window_1 - 1
    ],
    "lag_7": sales_matrix_window_1[
        series_rows_window_1, target_positions_window_1 - 7
    ],
    "lag_28": sales_matrix_window_1[
        series_rows_window_1, target_positions_window_1 - 28
    ],
})

# Add weekday and product/store information
weekday_lookup = calendar_weekdays.set_index("d")["weekday"]
target_days_window_1 = (
    train_sales_catboost_all_window_1.columns.to_numpy()[
        target_positions_window_1
    ]
)
X_train_catboost_all_window_1["weekday"] = (
    weekday_lookup.reindex(target_days_window_1).to_numpy()
)

metadata_columns = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
metadata_values = all_sales[metadata_columns].astype(str).to_numpy()

for column_number, column_name in enumerate(metadata_columns):
    X_train_catboost_all_window_1[column_name] = metadata_values[
        series_rows_window_1, column_number
    ]

X_train_catboost_all_window_1 = X_train_catboost_all_window_1[
    catboost_feature_columns
]

# Actual sales on the sampled target days
y_train_catboost_all_window_1 = sales_matrix_window_1[
    series_rows_window_1, target_positions_window_1
]

print(
    X_train_catboost_all_window_1.shape,
    y_train_catboost_all_window_1.shape
)

model_catboost_all_window_1 = catboost.CatBoostRegressor(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    loss_function="MAE",
    random_seed=50,
    verbose=25,
    thread_count=2,
    allow_writing_files=False
)

model_catboost_all_window_1.fit(
    X_train_catboost_all_window_1,
    y_train_catboost_all_window_1,
    cat_features=catboost_categorical_columns
)

In [ ]:
validation_sales_catboost_all_window_1 = daily_sales_all.iloc[:, -56:-28]

train_values_window_1 = train_sales_catboost_all_window_1.to_numpy(copy=False)
metadata_values = all_sales[metadata_columns].astype(str).to_numpy()
weekday_lookup = calendar_weekdays.set_index("d")["weekday"]

forecast_values_window_1 = np.zeros(
    (n_series_window_1, 28),
    dtype=np.float32
)

def get_lag_values_window_1(lag, step):
    previous_forecast_step = step - lag

    if previous_forecast_step < 0:
        return train_values_window_1[:, previous_forecast_step]

    return forecast_values_window_1[:, previous_forecast_step]


for step, day in enumerate(validation_sales_catboost_all_window_1.columns):
    future_features_window_1 = pd.DataFrame({
        "lag_1": get_lag_values_window_1(1, step),
        "lag_7": get_lag_values_window_1(7, step),
        "lag_28": get_lag_values_window_1(28, step),
        "weekday": np.repeat(weekday_lookup.loc[day], n_series_window_1)
    })

    for column_number, column_name in enumerate(metadata_columns):
        future_features_window_1[column_name] = metadata_values[
            :, column_number
        ]

    future_features_window_1 = future_features_window_1[
        catboost_feature_columns
    ]

    predicted_sales = model_catboost_all_window_1.predict(
        future_features_window_1
    )

    # Sales cannot be negative
    forecast_values_window_1[:, step] = np.maximum(predicted_sales, 0)

    if (step + 1) % 7 == 0:
        print(f"Forecasted {step + 1} of 28 days")


forecast_catboost_all_window_1 = pd.DataFrame(
    forecast_values_window_1,
    index=validation_sales_catboost_all_window_1.index,
    columns=validation_sales_catboost_all_window_1.columns
)

forecast_catboost_all_window_1.shape, forecast_catboost_all_window_1.iloc[:5, :5]

In [ ]:
# Repeat each series' average sales from its last 7 training days
avg_last_7_catboost_all_window_1 = (
    train_sales_catboost_all_window_1.iloc[:, -7:].mean(axis=1)
)

forecast_7day_catboost_all_window_1 = pd.DataFrame(
    np.repeat(
        avg_last_7_catboost_all_window_1.to_numpy()[:, None],
        validation_sales_catboost_all_window_1.shape[1],
        axis=1
    ),
    index=validation_sales_catboost_all_window_1.index,
    columns=validation_sales_catboost_all_window_1.columns
)

# Score both methods on the same validation data
mae_catboost_all_window_1 = np.abs(
    validation_sales_catboost_all_window_1.to_numpy()
    - forecast_catboost_all_window_1.to_numpy()
).mean()

mae_7day_all_window_1 = np.abs(
    validation_sales_catboost_all_window_1.to_numpy()
    - forecast_7day_catboost_all_window_1.to_numpy()
).mean()

comparison_all_window_1 = pd.DataFrame({
    "method": ["CatBoost with product/store info", "7-day average"],
    "MAE": [mae_catboost_all_window_1, mae_7day_all_window_1]
}).sort_values("MAE").reset_index(drop=True)

comparison_all_window_1

### Train CatBoost for window 2

For this window, the model will train only on data through d_1829. We’ll sample five historical target days per series, then test on d_1830–d_1857.

In [ ]:
# Window 2: train through d_1829
train_sales_catboost_all_window_2 = daily_sales_all.iloc[:, :-84]
sales_matrix_window_2 = train_sales_catboost_all_window_2.to_numpy(copy=False)

n_series_window_2, n_train_days_window_2 = sales_matrix_window_2.shape

rng = np.random.default_rng(50)
series_rows_window_2 = np.repeat(
    np.arange(n_series_window_2),
    samples_per_series
)
target_positions_window_2 = rng.integers(
    low=28,
    high=n_train_days_window_2,
    size=n_series_window_2 * samples_per_series
)

# Create the three lag features
X_train_catboost_all_window_2 = pd.DataFrame({
    "lag_1": sales_matrix_window_2[
        series_rows_window_2, target_positions_window_2 - 1
    ],
    "lag_7": sales_matrix_window_2[
        series_rows_window_2, target_positions_window_2 - 7
    ],
    "lag_28": sales_matrix_window_2[
        series_rows_window_2, target_positions_window_2 - 28
    ],
})

# Add weekday and product/store information
weekday_lookup = calendar_weekdays.set_index("d")["weekday"]
target_days_window_2 = (
    train_sales_catboost_all_window_2.columns.to_numpy()[
        target_positions_window_2
    ]
)
X_train_catboost_all_window_2["weekday"] = (
    weekday_lookup.reindex(target_days_window_2).to_numpy()
)

metadata_columns = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
metadata_values = all_sales[metadata_columns].astype(str).to_numpy()

for column_number, column_name in enumerate(metadata_columns):
    X_train_catboost_all_window_2[column_name] = metadata_values[
        series_rows_window_2, column_number
    ]

X_train_catboost_all_window_2 = X_train_catboost_all_window_2[
    catboost_feature_columns
]

# Actual sales on the sampled target days
y_train_catboost_all_window_2 = sales_matrix_window_2[
    series_rows_window_2, target_positions_window_2
]

print(
    X_train_catboost_all_window_2.shape,
    y_train_catboost_all_window_2.shape
)

model_catboost_all_window_2 = catboost.CatBoostRegressor(
    iterations=100,
    depth=5,
    learning_rate=0.1,
    loss_function="MAE",
    random_seed=50,
    verbose=25,
    thread_count=2,
    allow_writing_files=False
)

model_catboost_all_window_2.fit(
    X_train_catboost_all_window_2,
    y_train_catboost_all_window_2,
    cat_features=catboost_categorical_columns
)

In [ ]:
# Free the sampled training table; keep the model and historical sales
del X_train_catboost_all_window_2
del y_train_catboost_all_window_2
del series_rows_window_2
del target_positions_window_2
del target_days_window_2

validation_sales_catboost_all_window_2 = daily_sales_all.iloc[:, -84:-56]

train_values_window_2 = sales_matrix_window_2
metadata_values = all_sales[metadata_columns].astype(str).to_numpy()
weekday_lookup = calendar_weekdays.set_index("d")["weekday"]

forecast_values_window_2 = np.zeros(
    (n_series_window_2, 28),
    dtype=np.float32
)

def get_lag_values_window_2(lag, step):
    previous_forecast_step = step - lag

    if previous_forecast_step < 0:
        return train_values_window_2[:, previous_forecast_step]

    return forecast_values_window_2[:, previous_forecast_step]


for step, day in enumerate(validation_sales_catboost_all_window_2.columns):
    future_features_window_2 = pd.DataFrame({
        "lag_1": get_lag_values_window_2(1, step),
        "lag_7": get_lag_values_window_2(7, step),
        "lag_28": get_lag_values_window_2(28, step),
        "weekday": np.repeat(weekday_lookup.loc[day], n_series_window_2)
    })

    for column_number, column_name in enumerate(metadata_columns):
        future_features_window_2[column_name] = metadata_values[
            :, column_number
        ]

    future_features_window_2 = future_features_window_2[
        catboost_feature_columns
    ]

    predicted_sales = model_catboost_all_window_2.predict(
        future_features_window_2
    )
    forecast_values_window_2[:, step] = np.maximum(predicted_sales, 0)

    if (step + 1) % 7 == 0:
        print(f"Forecasted {step + 1} of 28 days")


forecast_catboost_all_window_2 = pd.DataFrame(
    forecast_values_window_2,
    index=validation_sales_catboost_all_window_2.index,
    columns=validation_sales_catboost_all_window_2.columns
)

forecast_catboost_all_window_2.shape, forecast_catboost_all_window_2.iloc[:5, :5]

In [ ]:
avg_last_7_catboost_all_window_2 = (
    train_sales_catboost_all_window_2.iloc[:, -7:].mean(axis=1)
)

forecast_7day_catboost_all_window_2 = pd.DataFrame(
    np.repeat(
        avg_last_7_catboost_all_window_2.to_numpy()[:, None],
        validation_sales_catboost_all_window_2.shape[1],
        axis=1
    ),
    index=validation_sales_catboost_all_window_2.index,
    columns=validation_sales_catboost_all_window_2.columns
)

mae_catboost_all_window_2 = np.abs(
    validation_sales_catboost_all_window_2.to_numpy()
    - forecast_catboost_all_window_2.to_numpy()
).mean()

mae_7day_all_window_2 = np.abs(
    validation_sales_catboost_all_window_2.to_numpy()
    - forecast_7day_catboost_all_window_2.to_numpy()
).mean()

comparison_all_window_2 = pd.DataFrame({
    "method": ["CatBoost with product/store info", "7-day average"],
    "MAE": [mae_catboost_all_window_2, mae_7day_all_window_2]
}).sort_values("MAE").reset_index(drop=True)

comparison_all_window_2

In [ ]:
print(
    "Training:",
    train_sales_catboost_all_window_2.shape,
    train_sales_catboost_all_window_2.columns[[0, -1]].tolist()
)

print(
    "Validation:",
    validation_sales_catboost_all_window_2.shape,
    validation_sales_catboost_all_window_2.columns[[0, -1]].tolist()
)